# B2-024 — Session 1: The GPU Workflow on Colab L4

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

Round 2's open-ended tasks are run in a hosted notebook that may give you an NVIDIA L4 GPU.
A GPU does not change *what* you compute; it changes where tensors live, which number formats are fast, how much memory you have, and what happens when the session dies.
This session teaches the habits that make one piece of code run correctly on a laptop CPU and fast on an L4: device-agnostic code, reading device-mismatch errors, autocast and dtype choice, memory arithmetic, out-of-memory triage, resumable checkpoints, and a written session plan.

Every code cell below runs on a CPU.
Cells that touch CUDA are guarded by `torch.cuda.is_available()`, and the text says which printed lines only appear on a GPU.
Book 1 already drew the line between CPU-only Round 1 work and accelerator work (`book1:cpu-and-gpu-round-boundary`) and taught the Colab submission routine (`book1:colab-coding-submission`); this session builds the GPU half of that picture.

## 1. Devices and device-agnostic code

Every tensor has a `device` attribute: `cpu`, `cuda:0` (the first GPU), or a few special devices.
An operation runs on the device of its inputs, and **all inputs must be on the same device**.
PyTorch never moves data for you; moving is an explicit, and comparatively slow, copy.

**The device-agnostic pattern.** Decide the device once, at the top of the script, and pass it everywhere:

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ShapeCNN().to(device)            # moves every parameter and buffer, in place
for images, labels in batches:
    images = images.to(device)           # returns a NEW tensor; you must keep it
    labels = labels.to(device)
    logits = model(images)
```

Two asymmetries cause most bugs:

* `module.to(device)` moves the module's registered parameters and buffers **in place** and returns the same module object, so `model.to(device)` on its own line works.
* `tensor.to(device)` returns a **new** tensor (or the same tensor if it is already there) and never changes the original, so `images.to(device)` on its own line does nothing useful. You must write `images = images.to(device)`.

**What counts as "the model".** Only tensors registered with the module move: parameters (`nn.Parameter`, created by layers such as `nn.Conv2d`) and buffers (`self.register_buffer(...)`).
A plain tensor attribute such as `self.mean = torch.tensor(...)` stays on the CPU when the model moves.
Book 1's `book1:nn-module` lesson showed registration for parameters; buffers are the same idea for fixed tensors (normalization statistics, constant grids) that must travel with the model and be saved in its `state_dict`, but receive no gradient.

**Creating tensors inside `forward`.** A tensor created with `torch.zeros(n)` inside `forward` lands on the default device (the CPU).
Create it on the input's device instead: `torch.zeros(n, device=x.device)` or `x.new_zeros(n)`.

**Moving whole batches.** Data loaders often return a tuple, list, or dict of tensors.
A small helper that walks the structure and moves every tensor (leaving labels-as-strings or integers alone) prevents the "I moved the images but not the labels" bug; practice p06 asks you to write it.

**Checkpoint 1A.** After `model.to(device)` and `x.to(device)` (both on their own lines) on a GPU machine, which of `model` and `x` is on the GPU?

**Checkpoint 1B.** A module stores `self.offset = torch.ones(3)` and adds it in `forward`. What goes wrong after `model.to('cuda')`, and what is the fix?

**Collected answers.** `model` is on the GPU (module `.to` moves registered tensors in place); `x` is still on the CPU because tensor `.to` returned a new tensor that was discarded. The plain attribute stays on the CPU, so `forward` mixes a CUDA activation with a CPU tensor and raises a device-mismatch error; register it with `self.register_buffer("offset", torch.ones(3))` so it moves with the module (and is saved in the `state_dict`).

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device chosen:", device)

torch.manual_seed(SEED)
model = capstone_data.ShapeCNN()
returned = model.to(device)
print("module .to returns the same object:", returned is model)

images, labels = capstone_data.train_rows("shapes_supervised")
batch = images[:4]
moved = batch.to(device)
print("tensor .to returned the same object:", moved is batch,
      "(True only when the tensor is already on that device)")
logits = model(moved)
print("logits:", tuple(logits.shape), logits.device, logits.dtype)

## 2. Reading device-mismatch errors

A device mismatch raises a `RuntimeError` at the first operation that combines tensors from two devices.
On a GPU the message reads like

```text
RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!
(when checking argument for argument mat1 in method wrapper_CUDA_addmm)
```

Read it in three parts:

1. **The two devices** (`cuda:0` and `cpu`): one operand was not moved.
2. **The operation** (`addmm` is the matrix multiply-add inside `nn.Linear`; a convolution shows `convolution`): this tells you *which layer* failed.
3. **The argument** (`mat1` is the input batch of `addmm`; `mat2` would be the weight): this tells you *which side* is on the wrong device.

So `mat1 ... addmm` with `cuda:0 and cpu` means the *input* to a linear layer is on the CPU while the weight is on the GPU: the batch (or an activation created inside `forward`) was not moved.

**A triage checklist**, in order: (1) did every batch tensor get reassigned after `.to(device)`; (2) was the model moved *before* the optimizer was created (an optimizer built from CPU parameters keeps state on the CPU); (3) is any tensor created inside `forward` or the loss without `device=`; (4) is any constant stored as a plain attribute instead of a buffer; (5) are the targets on the same device as the logits for the loss.

**Seeing a real mismatch on a CPU-only machine.** PyTorch has a `meta` device whose tensors carry shape and dtype but no data.
Moving a layer to `meta` and feeding it a CPU tensor raises the same class of error, so the cell below demonstrates the failure without a GPU.
The wording differs slightly from the CUDA message; the diagnosis is identical.

**Checkpoint 2A.** The message says `found at least two devices, cuda:0 and cpu` while checking argument `mat2` in `addmm`. Which tensor is on the wrong device?

**Checkpoint 2B.** Why must the model be moved to the GPU before `torch.optim.Adam(model.parameters())` is created?

**Collected answers.** `mat2` is the weight side of the linear layer's multiply, so a layer's parameter is on the CPU while its input is on the GPU: some part of the model was not moved (often a plain-attribute tensor or a layer created after `model.to(device)`). The optimizer stores references to the parameter tensors and creates its moment buffers on their devices; moving first means the optimizer sees the GPU parameters (in current PyTorch `module.to` keeps the same parameter objects, but creating the optimizer after the move is the safe, conventional order and avoids state on the wrong device after loading checkpoints).

In [ ]:
layer = nn.Linear(4, 2).to("meta")        # a stand-in "second device" with no data
cpu_batch = torch.zeros(3, 4)
try:
    layer(cpu_batch)
except RuntimeError as err:
    print("RuntimeError:", str(err).splitlines()[0])

fixed = layer.to_empty(device="cpu")        # materialize on the CPU (values uninitialized)
print("after moving the layer to the batch's device:", tuple(fixed(cpu_batch).shape))

## 3. Mixed precision: dtypes and autocast

A float format trades range and precision for size and speed:

| dtype | bits | exponent bits | mantissa bits | largest finite value | typical use |
|---|---:|---:|---:|---:|---|
| `float32` | 32 | 8 | 23 | about $3.4\times10^{38}$ | master weights, CPU work, the CI path |
| `bfloat16` | 16 | 8 | 7 | about $3.4\times10^{38}$ | activations and matmuls on L4 / A100 / H100 |
| `float16` | 16 | 5 | 10 | $65{,}504$ | activations on older GPUs, needs loss scaling |

`bfloat16` keeps `float32`'s exponent range and gives up precision; `float16` keeps more precision but overflows above $65{,}504$ and underflows small gradients to zero.

**Autocast.** `torch.autocast(device_type, dtype)` is a context manager that runs selected operations (matrix multiplies, convolutions) in the lower precision while keeping numerically sensitive ones (softmax, losses, reductions) in `float32`.
The **parameters stay `float32`** ("master weights"), gradients flow back in `float32`, and the optimizer updates `float32` numbers.
Only the forward pass (and the matching backward computations) goes inside the context:

```python
with torch.autocast(device_type=device.type, dtype=amp_dtype):   # skip entirely if amp_dtype is None
    logits = model(images)
    loss = F.cross_entropy(logits, labels)
loss.backward()            # outside the context
optimizer.step()
```

**Which dtype on an L4.** The L4 (Ada generation) has fast `bfloat16` tensor cores, so `amp_dtype=torch.bfloat16` is the default choice: no loss scaler is needed because the exponent range matches `float32`.
`float16` is the alternative on GPUs without `bfloat16`; it needs `torch.amp.GradScaler`, which multiplies the loss by a large factor before `backward()` so small gradients do not underflow, then unscales before `step()` and skips steps whose gradients overflowed.

**The CI path is `amp_dtype=None`.** CI runs on a CPU, in `float32`, with no autocast, because that is the reproducible reference.
Running autocast with `bfloat16` **on a CPU** is legal and useful for *correctness* (it proves your code path handles a lower-precision activation dtype), but it is not a speed test: on CPUs without native `bfloat16` arithmetic it can be many times slower than `float32`.
The cell below shows the dtypes under each setting, then times a tiny matrix multiply in both formats on this machine; read the timing as "it can be slower", not as a benchmark.

**Checkpoint 3A.** Inside `torch.autocast('cpu', dtype=torch.bfloat16)`, what dtype do the ShapeCNN logits have, and what dtype do the convolution weights keep?

**Checkpoint 3B.** Why does `bfloat16` training not need a gradient scaler while `float16` training does?

**Collected answers.** The logits are `bfloat16` (the head's matrix multiply ran in lower precision); the weights stay `float32` master copies, and their gradients are `float32`. `bfloat16` has the same 8 exponent bits as `float32`, so small gradients do not underflow to zero; `float16` has only 5 exponent bits (smallest normal value about $6\times10^{-5}$), so tiny gradients vanish unless the loss is scaled up first.

In [ ]:
import time

torch.manual_seed(SEED)
model = capstone_data.ShapeCNN()
images, labels = capstone_data.train_rows("shapes_supervised")
x, y = images[:16], labels[:16]
for amp_dtype in (None, torch.bfloat16):
    model.zero_grad(set_to_none=True)
    if amp_dtype is None:
        logits = model(x)
        loss = F.cross_entropy(logits, y)
    else:
        with torch.autocast(device_type="cpu", dtype=amp_dtype):
            logits = model(x)
            loss = F.cross_entropy(logits, y)
    loss.backward()
    print(f"amp_dtype={amp_dtype}: logits {logits.dtype}, loss {loss.dtype}, "
          f"weight {model.conv1.weight.dtype}, grad {model.conv1.weight.grad.dtype}")

a = torch.randn(128, 128)
for dtype in (torch.float32, torch.bfloat16):
    m = a.to(dtype)
    start = time.perf_counter()
    for _ in range(5):
        m @ m
    print(f"{dtype}: {1000 * (time.perf_counter() - start):.2f} ms for five 128x128 matmuls on this CPU")

if torch.cuda.is_available():   # this branch only prints on a GPU machine
    print("GPU bf16 supported:", torch.cuda.is_bf16_supported())

## 4. Memory arithmetic and batch sizing

GPU memory is the binding constraint on an L4 (24 GB).
Estimate it before you run, using `book1:parameter-counting`.

**Bytes per element:** `float32` 4, `bfloat16`/`float16` 2, `int64` 8.

**Training with Adam in `float32` master precision** stores, per parameter: the weight (4 bytes), its gradient (4 bytes), and Adam's two moment estimates $m$ and $v$ (4 + 4 bytes): **16 bytes per parameter**.
Mixed precision does not shrink these four tensors: the master weights, gradients, and moments stay `float32`.

**Activations** are the intermediate tensors the forward pass keeps for `backward()`.
Their count scales with the batch size $B$: if one example produces $e$ activation elements, a batch keeps about $Be$ of them, at 4 bytes each in `float32` or 2 bytes each under `bfloat16` autocast.

$$\text{training memory} \approx 16P + (\text{bytes per activation})\cdot B e .$$

**Worked example.** A model has $P = 2{,}000{,}000$ parameters and keeps $e = 50{,}000$ activation elements per example.
Parameters, gradients and Adam states: $16P = 32{,}000{,}000$ bytes $= 32$ MB.
At $B=256$ in `float32`: activations $4\cdot256\cdot50{,}000 = 51.2$ MB, total about $83$ MB.
Under `bfloat16` autocast: activations $25.6$ MB, total about $58$ MB, a ratio $58/83\approx0.70$, not $0.5$, because the 16 bytes per parameter did not shrink.

**Batch sizing.** Solve for the largest $B$ that fits a *safety budget* (say 70% of the card, leaving room for the CUDA context, temporary buffers, and fragmentation):
$$B_{\max} = \left\lfloor\frac{0.7\cdot M - 16P}{(\text{bytes per activation})\cdot e}\right\rfloor .$$
Then measure: on a GPU, `torch.cuda.max_memory_allocated()` after one training step reports the true peak, which you compare with the estimate before scaling up.
The estimate is a planning tool, not a guarantee; the measurement is the truth.

The cell below counts ShapeCNN's parameters and activation elements exactly (by recording every intermediate tensor in one forward pass) and turns them into the formula's numbers.

**Checkpoint 4A.** A model has $P=10^6$ parameters and $e=10^5$ activation elements per example. Estimate training memory at $B=100$ in all-`float32` and with `bfloat16` activations.

**Checkpoint 4B.** Why does halving the activation precision never halve total training memory in this model of memory?

**Collected answers.** $16P = 16$ MB; activations $4\cdot100\cdot10^5=40$ MB (`float32`) or $20$ MB (`bfloat16`); totals $56$ MB and $36$ MB. Because the $16P$ bytes of `float32` weights, gradients and two Adam moments are unchanged; only the activation term halves, so the ratio is $(16P+2Be)/(16P+4Be)>1/2$.

In [ ]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

def activation_elements(model, x):
    """Elements of every intermediate the forward keeps (conv/linear outputs and ReLU outputs)."""
    total = 0
    h1 = model.conv1(x); total += h1.numel()
    a1 = F.relu(h1); total += a1.numel()
    h2 = model.conv2(a1); total += h2.numel()
    a2 = F.relu(h2); total += a2.numel()
    out = model.head(a2.flatten(1)); total += out.numel()
    return total

torch.manual_seed(SEED)
cnn = capstone_data.ShapeCNN()
P = count_params(cnn)
e = activation_elements(cnn, torch.zeros(1, 1, 8, 8))
print("ShapeCNN parameters P =", P, "| activation elements per example e =", e)
for B in (64, 512):
    fp32 = 16 * P + 4 * B * e
    bf16 = 16 * P + 2 * B * e
    print(f"B={B}: fp32 {fp32/1e6:.3f} MB, bf16-activations {bf16/1e6:.3f} MB, ratio {bf16/fp32:.3f}")

budget = 0.7 * 24e9
print("L4 70% budget, bf16 activations: B_max =", int((budget - 16 * P) // (2 * e)))
if torch.cuda.is_available():   # GPU-only measurement
    torch.cuda.reset_peak_memory_stats()
    print("measured peak after one step would be read with torch.cuda.max_memory_allocated()")

## 5. Out-of-memory triage

`torch.OutOfMemoryError: CUDA out of memory. Tried to allocate ...` means the *peak* demand exceeded what the allocator could find.
Triage in this order, cheapest and most likely first:

1. **Check for a leak before shrinking anything.** The classic leak is `history.append(loss)` instead of `history.append(loss.item())`: each stored loss tensor keeps its whole computation graph (all activations) alive. Memory then grows step after step until it fails. Also check evaluation code runs under `torch.no_grad()`, which stops the graph from being built at all.
2. **Lower the activation precision.** Switch from `amp_dtype=None` to `bfloat16` autocast on an L4; this roughly halves the activation term.
3. **Reduce the batch size.** Activation memory is linear in $B$. If the training recipe needs the larger batch, keep the *effective* batch with **gradient accumulation**: run $k$ smaller batches, call `backward()` on each (gradients add up), divide the loss by $k$, and call `optimizer.step()` once.
4. **Shrink the model or the input** (width, resolution) only if the experiment allows it; this changes the experiment, so log it.
5. **Release cached memory last.** `torch.cuda.empty_cache()` returns *unused cached* blocks to the driver; it cannot free tensors you still reference, so it fixes fragmentation symptoms, not leaks.

After an OOM inside a notebook, the failed step's tensors may still be referenced by the traceback.
The reliable recovery is: save what you need (a checkpoint, Section 6), restart the runtime, fix the cause, and resume.
That is why resumable training is part of the GPU workflow rather than an optional nicety.

**Measuring rather than guessing.** On a GPU, `torch.cuda.max_memory_allocated()` (peak bytes held by tensors) and `torch.cuda.memory_reserved()` (bytes held by the caching allocator) after one step tell you which term dominates.
If the peak grows from step 1 to step 50 at a fixed batch size, you have a leak (item 1); if it is flat but too large, you need items 2–4.

**Checkpoint 5A.** Peak memory rises by the same amount every training step at a fixed batch size. Which triage item comes first, and what is the most likely line to look for?

**Checkpoint 5B.** You need an effective batch of 512 but only 128 fits. Write the update schedule with gradient accumulation.

**Collected answers.** Item 1 (a leak): look for a stored tensor that keeps the graph alive, typically `losses.append(loss)` (fix: `loss.item()`), or evaluation without `torch.no_grad()`. For each of 4 micro-batches of 128: forward, compute `loss / 4`, call `backward()` (gradients accumulate); after the fourth, call `optimizer.step()` and then `optimizer.zero_grad(set_to_none=True)`.

## 6. Resumable training: checkpoints with RNG state

Hosted sessions disconnect.
A run that cannot resume loses all its work; a run that resumes *differently* silently changes the experiment.
A complete checkpoint holds everything the next step depends on:

* `model.state_dict()` — parameters and buffers;
* `optimizer.state_dict()` — Adam's step counts and moment estimates (resuming with a fresh optimizer restarts Adam's bias correction and changes every later step);
* the integer `step`;
* the **random-number-generator state**: `torch.get_rng_state()` for the CPU generator, plus `torch.cuda.get_rng_state_all()` when CUDA is present.

**Why the RNG state matters.** This unit's training loops sample mini-batches from the *global* torch generator, for example `idx = torch.randint(0, n, (batch_size,))`.
After a restart the generator is re-seeded or fresh, so without restoring its state the resumed run draws different batches from the uninterrupted run.
The model still trains, but it is no longer the same experiment, and the "resume" cannot be verified.
Restoring the state with `torch.set_rng_state(...)` (and `torch.cuda.set_rng_state_all(...)` when present) makes step $k+1$ of the resumed run draw exactly the batch the uninterrupted run drew.

**The certificate.** Run the uninterrupted loop to step $k+1$ and record its loss.
Separately, run to step $k$, save, build a *fresh* model and optimizer, load, and run step $k+1$.
On a CPU with the same thread settings the two losses are bit-for-bit identical, so the check is `torch.equal`, not `allclose`.
(On a GPU, some kernels are nondeterministic; there you would compare with a tolerance or enable deterministic algorithms.)

**Saving to persistent storage.** On Colab, the runtime's disk disappears with the session; checkpoints must go to a mounted drive or be downloaded.
Save to a temporary file first and rename it, so a disconnect during the write never leaves a half-written "latest" checkpoint.

**Checkpoint 6A.** A resumed run loads model and optimizer state but not the RNG state. Will its loss at step $k+1$ match the uninterrupted run? Why?

**Checkpoint 6B.** Why is `torch.equal` (not a tolerance) the right comparison for the CPU resume certificate?

**Collected answers.** Generally no: the global generator is in a different state, so `torch.randint` draws a different mini-batch at step $k+1$, giving a different loss (the weights are identical; the data is not). On a CPU with fixed threads every operation is deterministic, so identical inputs, weights and optimizer state produce bit-identical results; any difference signals a real bug (missing state), which a tolerance could hide.

## 7. The Colab L4 session plan

A GPU session is a scarce, interruptible resource, so plan it on paper before you connect.
The plan has five parts.

**1. Time budget.** Write the wall-clock budget for the session and split it: setup and smoke test (about 10%), the main run (about 60%), evaluation and writeup (about 20%), slack for one restart (about 10%).
Estimate the main run from a measured CPU or GPU steps-per-second figure, never from hope; if the estimate does not fit, shrink the config before connecting.

**2. Smoke test first.** Run the CPU config (or a tiny GPU config) end-to-end on the GPU runtime: data loading, one training step under `bfloat16` autocast, one checkpoint save and load, one validation pass.
This catches device mismatches and path errors in a minute instead of after an hour.

**3. Checkpoint cadence.** Choose the interval so that a disconnect loses at most a small, stated fraction of the budget (for example 5 minutes of work), and keep the latest checkpoint plus the best-on-validation one.
Writing a checkpoint costs time too; for a few-MB model it is negligible, for a multi-GB model the interval should be longer.

**4. Restart procedure** (write it down as numbered steps): reconnect; re-run the setup cell (imports, the data module, the config); mount persistent storage; load `latest.pt` including RNG state; check that the step counter and the last logged validation score match the log; continue.
Your experiment log (Session 2) records each restart.

**5. Submission checklist** (extending Book 1's `book1:colab-coding-submission`): the notebook runs top to bottom on the CPU config without the GPU; the GPU-only cells are guarded by `torch.cuda.is_available()`; seeds and the config dict are printed; the locked test set was scored exactly once; required files (predictions, writeup) are saved where the grader expects them; no checkpoint or download the rules forbid is referenced.

Book 1's `book1:cpu-and-gpu-round-boundary` stated that Round 1 is CPU-only while Round 2 may grant an accelerator.
The plan above is how you turn that grant into finished, reproducible work instead of a half-trained model and an expired session.

**Checkpoint 7A.** Your measured throughput is 40 steps/s and the main-run share of a 2-hour session is 60%. What is the largest step count you can plan for?

**Checkpoint 7B.** List the first three actions of the restart procedure.

**Collected answers.** $0.6\cdot7200\text{ s}=4320$ s, times 40 steps/s $=172{,}800$ steps (then leave margin for checkpoints and evaluation). Reconnect to the runtime; re-run the setup cell (imports, data module, config); mount persistent storage (then load `latest.pt` with its RNG state and verify the step counter against the log).

## 8. Worked laboratory: one config, two machines, an exact resume

The laboratory puts Sections 1–6 together on the `shapes_supervised` training rows.
One **config dict** drives one code path.
The CPU config is what this notebook (and CI) runs; the L4 config is the same dictionary with larger values and is shown, not executed.
Because both go through the same function, the accelerator version cannot drift from the code that was tested.

The loop samples batches with the global generator, checkpoints at step `k`, then proves that a fresh model restored from the checkpoint reproduces the uninterrupted run's next loss exactly.
A wrong variant that skips the RNG restore is run too, to show that it draws a different batch.

**Checkpoint 8A.** Which two config entries would you change first to use an L4, and which entry must stay `None` for the CI path?

**Checkpoint 8B.** In the laboratory, why does the no-RNG variant still produce a finite, plausible loss even though it is wrong?

**Collected answers.** `device` (to `"cuda"`) and `amp_dtype` (to `torch.bfloat16`), then batch size and steps; `amp_dtype` stays `None` on the CPU path. Its weights and optimizer state are correct, so it trains normally; only the mini-batch differs, which changes the loss value without any error, which is exactly why the exact-equality certificate is needed.

In [ ]:
import os
import tempfile

CPU_CONFIG = {"device": "cpu", "amp_dtype": None, "batch_size": 64, "steps": 24, "checkpoint_at": 12, "lr": 0.01}
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "batch_size": 512, "steps": 4000, "checkpoint_at": 500, "lr": 0.01}

def make_run(config):
    torch.manual_seed(SEED)
    model = capstone_data.ShapeCNN().to(config["device"])
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    return model, optimizer

def one_step(model, optimizer, x_all, y_all, config):
    idx = torch.randint(0, x_all.shape[0], (config["batch_size"],))     # global RNG
    x = x_all[idx].to(config["device"])
    y = y_all[idx].to(config["device"])
    optimizer.zero_grad(set_to_none=True)
    if config["amp_dtype"] is None:
        loss = F.cross_entropy(model(x), y)
    else:
        with torch.autocast(device_type=torch.device(config["device"]).type, dtype=config["amp_dtype"]):
            loss = F.cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
    return loss.detach()

def save(path, model, optimizer, step):
    state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": step,
             "torch_rng": torch.get_rng_state(),
             "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
    torch.save(state, path + ".tmp")
    os.replace(path + ".tmp", path)          # atomic rename: never a half-written checkpoint

def load(path, model, optimizer, restore_rng=True):
    state = torch.load(path)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    if restore_rng:
        torch.set_rng_state(state["torch_rng"])
        if state["cuda_rng"] is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(state["cuda_rng"])
    return state["step"]

In [ ]:
cfg = CPU_CONFIG
x_all, y_all = capstone_data.train_rows("shapes_supervised")
k = cfg["checkpoint_at"]

model, optimizer = make_run(cfg)                      # seeds the global RNG
trace = [one_step(model, optimizer, x_all, y_all, cfg) for _ in range(k + 1)]
uninterrupted_next = trace[k]                         # loss of step k+1 (index k)

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "latest.pt")
    model, optimizer = make_run(cfg)
    for _ in range(k):
        one_step(model, optimizer, x_all, y_all, cfg)
    save(path, model, optimizer, k)

    fresh_model, fresh_opt = make_run(cfg)            # also re-seeds: simulates a restarted session
    torch.manual_seed(12345)                          # the new session's RNG is somewhere else
    assert load(path, fresh_model, fresh_opt) == k
    resumed_next = one_step(fresh_model, fresh_opt, x_all, y_all, cfg)

    wrong_model, wrong_opt = make_run(cfg)
    torch.manual_seed(12345)
    load(path, wrong_model, wrong_opt, restore_rng=False)
    wrong_next = one_step(wrong_model, wrong_opt, x_all, y_all, cfg)

print("first loss", round(trace[0].item(), 4), "| loss at step k+1", round(uninterrupted_next.item(), 4))
print("resumed equals uninterrupted (torch.equal):", torch.equal(resumed_next, uninterrupted_next))
print("no-RNG-restore equals uninterrupted:", torch.equal(wrong_next, uninterrupted_next), "| its loss", round(wrong_next.item(), 4))
assert torch.equal(resumed_next, uninterrupted_next)
print("L4 config (shown, not run here):", L4_CONFIG)

## 9. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *`x.to(device)` without assignment.* Fix: `x = x.to(device)`; module `.to` is in place, tensor `.to` is not.
2. *Constants as plain attributes.* Fix: `register_buffer`, so they move and are saved.
3. *Tensors created in `forward` without a device.* Fix: `device=x.device` or `x.new_zeros(...)`.
4. *Autocast around `backward()` or the optimizer step.* Fix: only the forward pass and loss go inside.
5. *Benchmarking `bfloat16` on a CPU.* Fix: use CPU `bfloat16` for correctness only; measure speed on the GPU.
6. *Expecting mixed precision to halve memory.* Fix: the $16P$ bytes of `float32` state stay; recompute with the formula.
7. *Storing loss tensors in a list.* Fix: `.item()`; evaluate under `torch.no_grad()`.
8. *Checkpoints without optimizer or RNG state.* Fix: save all four parts; certify the resume with `torch.equal` on the CPU.
9. *Saving checkpoints to the runtime's ephemeral disk.* Fix: persistent storage, atomic rename.

**Exam connections.** p01 computes a mixed-precision memory ratio in normal form; p02 diagnoses a device-mismatch traceback; p06 implements `get_device`, `move_batch`, and an autocast `train_step` checked on the CPU; p07 implements checkpoints with RNG state and proves an exact resume; p16 trains a small CNN with those helpers and certifies loss decrease and resume equality; p23 writes a complete Colab L4 session plan.
In Round 2, a task that offers a GPU is graded on the submitted notebook's results and its reproducibility, so these habits protect points rather than earn style marks.

**Checkpoint 9A.** Which pitfall explains a model that trains fine on the CPU but raises a mismatch on the GPU only when a normalization constant is used?

**Checkpoint 9B.** Which practice asks you to prove an exact resume, and what comparison does it use?

**Collected answers.** Pitfall 2: the constant is a plain attribute that did not move with `model.to(device)`; register it as a buffer. p07 (and p16 at training scale), with `torch.equal` on the resumed next-step loss.

## 10. Forward-only deeper topic: running a real latent-diffusion pipeline on a Colab L4

Forward-only deeper topic.
Session 5 of B2-023 drew the Stable Diffusion component map and traced a CPU-scale class-conditional latent diffusion model, and promised this hand-off.
Real text-to-image latent-diffusion systems use the same parts (an autoencoder that maps images to a small latent grid, a large U-Net denoiser, a text encoder whose tokens condition the denoiser through cross-attention, and a scheduler) but with on the order of a billion parameters and pretrained weights.
This guide describes how such a pipeline is typically run on a Colab L4.
It is **markdown only**: there is no code cell, nothing here is executed or assessed, nothing in this unit depends on it, and CI never downloads anything.

**Before you start.**
Pretrained checkpoints are distributed by model hubs under licenses and acceptable-use terms; read the model card and license before downloading, and do not use such downloads in any task whose rules forbid them (this unit's practices forbid them).
A typical latent-diffusion checkpoint is several gigabytes, so the download alone can take minutes of your session budget.

**The session, step by step.**

1. Select a GPU runtime with an L4 and confirm it: the device name reported by the runtime and `torch.cuda.is_available()` should both indicate the GPU.
2. Install a diffusion pipeline library in the notebook environment, pinning versions so the run is reproducible.
3. Load the pipeline with half-precision weights (`bfloat16` on an L4, or `float16`) and move the whole pipeline to `cuda` with one `.to(...)` call; the memory arithmetic of Section 4 tells you why: at 2 bytes per parameter a one-billion-parameter denoiser needs about 2 GB for weights alone, before activations, and inference keeps no gradients or Adam moments.
4. Create a `torch.Generator(device="cuda")` with a fixed seed and pass it to the pipeline call, so the initial latent noise, and therefore the image, is reproducible.
5. Generate one image at a small resolution and a modest number of denoising steps first (a smoke test), then scale up. Observe three things: the time per denoising step, the peak memory (Section 4's measurement), and how the image changes as the number of steps and the guidance scale change.
6. Save outputs to persistent storage (Sections 6 and 7), with the seed, step count, guidance scale, and library versions written next to them.

**What to connect it to.**
The denoising loop is B2-023's reverse diffusion with its scheduler; the autoencoder is B2-022's encoder–decoder; classifier-free guidance (B2-023) mixes a conditional and an unconditional noise prediction; the memory and session-planning habits are this session's.

This final section is forward-only and not assessed.

**Checkpoint 10A.** Why does inference with a pretrained pipeline need far less memory per parameter than training in this session's memory model?

**Checkpoint 10B.** Which object makes a pipeline's output image reproducible, and where is it passed?

**Collected answers.** Inference stores only the (half-precision) weights and transient activations: no gradients and no Adam moments, so about 2 bytes per parameter instead of 16 plus activations kept for backward. A seeded `torch.Generator` (on the GPU device), passed to the pipeline call so the initial latent noise is fixed.